# 1. Alibaba reranker

In [ ]:
import re
import chromadb
import torch
import pandas as pd
from collections import defaultdict
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from konlpy.tag import Okt
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# =========================
# 설정
# =========================
PERSIST_DIR = "chroma_store"
COLLECTION_NAME = "laws_by_article"

EMBEDDING_MODEL = "jinaai/jina-embeddings-v3"
RERANKER_MODEL = "Alibaba-NLP/gte-multilingual-reranker-base"

DEVICE = "mps"   # 안되면 "cpu"
TOP_K_DENSE = 10
TOP_K_BM25 = 10
TOP_K_RRF = 10
TOP_K_FINAL = 5
RRF_K = 60

# =========================
# 디바이스 설정
# =========================
if DEVICE == "mps" and torch.backends.mps.is_available():
    torch_device = torch.device("mps")
else:
    torch_device = torch.device("cpu")

# =========================
# 형태소 분석기
# =========================
okt = Okt()

STOPWORDS = {
    "기준", "여부", "가능", "얼마", "등", "통해", "경우", "사항"
}

COMPOUND_TERMS = [
    "어린이보호구역",
    "어린이 보호구역",
    "보호구역",
    "제한속도",
    "속도제한",
    "최고속도",
    "통행속도",
    "스쿨존",
    "속도위반",
    "주정차위반",
    "주정차 위반",
    "주차위반",
    "정차위반",
    "과태료",
    "범칙금",
    "부과기준",
    "면허취소",
    "면허정지",
    "운전면허",
    "음주운전",
    "취소처분",
    "주차단속",
    "불법주차",
    "신고",
    "견인",
    "조치",
]

COMPOUND_TERMS_NORMALIZED = {
    term.replace(" ", ""): term.replace(" ", "") for term in COMPOUND_TERMS
}

# =========================
# BM25 전처리
# =========================
def custom_preprocess(text: str):
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    pos_tags = okt.pos(text, norm=True, stem=True)

    base_tokens = []
    for word, pos in pos_tags:
        if pos in ["Noun", "Alpha", "Number"] and len(word) >= 2:
            base_tokens.append(word)

    compact_text = re.sub(r"\s+", "", text)

    compound_tokens = []
    for compact_term in COMPOUND_TERMS_NORMALIZED:
        if compact_term in compact_text:
            compound_tokens.append(compact_term)

    tokens = base_tokens + compound_tokens

    cleaned = []
    seen = set()

    for token in tokens:
        token = token.strip()
        if not token:
            continue
        if token in STOPWORDS:
            continue
        if len(token) < 2:
            continue
        if token not in seen:
            seen.add(token)
            cleaned.append(token)

    return cleaned

# =========================
# Dense embedding wrapper
# =========================
class SentenceTransformerEmbedding:
    def __init__(self, model_name=EMBEDDING_MODEL, device=DEVICE):
        self.model = SentenceTransformer(
            model_name,
            trust_remote_code=True,
            device=device
        )

    def embed_documents(self, texts):
        return self.model.encode(
            texts,
            batch_size=64,
            normalize_embeddings=True
        ).tolist()

    def embed_query(self, text):
        return self.model.encode(
            [text],
            normalize_embeddings=True
        )[0].tolist()

# =========================
# Chroma 연결
# =========================
embedding_fn = SentenceTransformerEmbedding()
client = chromadb.PersistentClient(path=PERSIST_DIR)

vectorstore = Chroma(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding_function=embedding_fn,
)

# =========================
# Dense Retriever
# =========================
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K_DENSE})

# =========================
# BM25용 문서 불러오기
# =========================
raw_data = vectorstore.get(include=["documents", "metadatas"])
documents = raw_data["documents"]
metadatas = raw_data["metadatas"]

docs = []
for doc, meta in zip(documents, metadatas):
    if doc and str(doc).strip():
        docs.append(
            Document(
                page_content=doc,
                metadata=meta if meta else {}
            )
        )

# =========================
# BM25 Retriever
# =========================
bm25_retriever = BM25Retriever.from_documents(
    docs,
    preprocess_func=custom_preprocess
)
bm25_retriever.k = TOP_K_BM25

# =========================
# Alibaba reranker 로드
# =========================
reranker_tokenizer = AutoTokenizer.from_pretrained(RERANKER_MODEL)
reranker_model = AutoModelForSequenceClassification.from_pretrained(RERANKER_MODEL)
reranker_model.to(torch_device)
reranker_model.eval()

# =========================
# reranker score 함수
# =========================
def get_reranker_scores(query, passages, batch_size=8, max_length=512):
    all_scores = []

    for start in range(0, len(passages), batch_size):
        batch_passages = passages[start:start + batch_size]
        pairs = [[query, passage] for passage in batch_passages]

        with torch.no_grad():
            inputs = reranker_tokenizer(
                pairs,
                padding=True,
                truncation=True,
                return_tensors="pt",
                max_length=max_length
            )
            inputs = {k: v.to(torch_device) for k, v in inputs.items()}
            logits = reranker_model(**inputs).logits.view(-1).float().cpu().tolist()
            all_scores.extend(logits)

    return all_scores

# =========================
# 질의
# =========================
queries = [
    "어린이 보호구역 제한속도는 얼마인가?",
    "스쿨존 속도위반 기준은?",
    "주정차 위반 과태료 기준은?",
    "음주운전 면허취소 기준은?",
    "불법주차 신고 가능 여부는?"
]

# =========================
# 결과 저장
# =========================
comparison_results = {}
rows = []

# =========================
# 검색 + RRF + reranker
# =========================
for query in queries:
    dense_results = dense_retriever.invoke(query)
    bm25_results = bm25_retriever.invoke(query)

    # -------------------------
    # 1) RRF 후보 생성
    # -------------------------
    rrf_scores = defaultdict(float)
    doc_store = {}
    dense_rank_map = {}
    bm25_rank_map = {}

    for rank, doc in enumerate(dense_results, start=1):
        doc_key = (
            doc.page_content.strip(),
            tuple(sorted(doc.metadata.items())) if doc.metadata else ()
        )
        rrf_scores[doc_key] += 1 / (RRF_K + rank)
        doc_store[doc_key] = doc
        dense_rank_map[doc_key] = rank

    for rank, doc in enumerate(bm25_results, start=1):
        doc_key = (
            doc.page_content.strip(),
            tuple(sorted(doc.metadata.items())) if doc.metadata else ()
        )
        rrf_scores[doc_key] += 1 / (RRF_K + rank)
        doc_store[doc_key] = doc
        bm25_rank_map[doc_key] = rank

    sorted_rrf = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    rrf_candidates = sorted_rrf[:TOP_K_RRF]

    rrf_docs = []
    for rrf_rank, (doc_key, rrf_score) in enumerate(rrf_candidates, start=1):
        doc = doc_store[doc_key]
        rrf_docs.append({
            "doc_key": doc_key,
            "doc": doc,
            "rrf_score": rrf_score,
            "rrf_rank": rrf_rank,
            "dense_rank": dense_rank_map.get(doc_key),
            "bm25_rank": bm25_rank_map.get(doc_key),
        })

    # -------------------------
    # 2) reranker 점수 계산
    # -------------------------
    passages = [item["doc"].page_content for item in rrf_docs]
    rerank_scores = get_reranker_scores(
        query=query,
        passages=passages,
        batch_size=8,
        max_length=512
    )

    for item, rerank_score in zip(rrf_docs, rerank_scores):
        item["rerank_score"] = rerank_score

    reranked_items = sorted(
        rrf_docs,
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    final_results = reranked_items[:TOP_K_FINAL]

    comparison_results[query] = {
        "query_tokens": custom_preprocess(query),
        "dense": [],
        "bm25_compound": [],
        "hybrid_rrf": [],
        "reranked": []
    }

    # -------------------------
    # Dense 저장
    # -------------------------
    for i, doc in enumerate(dense_results, start=1):
        comparison_results[query]["dense"].append({
            "rank": i,
            "metadata": doc.metadata,
            "page_content": doc.page_content
        })

        rows.append({
            "query": query,
            "stage": "dense",
            "rank": i,
            "rrf_score": None,
            "rerank_score": None,
            "dense_rank": i,
            "bm25_rank": None,
            "rrf_rank": None,
            "query_tokens": ", ".join(custom_preprocess(query)),
            "metadata": str(doc.metadata),
            "page_content": doc.page_content
        })

    # -------------------------
    # BM25 저장
    # -------------------------
    for i, doc in enumerate(bm25_results, start=1):
        comparison_results[query]["bm25_compound"].append({
            "rank": i,
            "metadata": doc.metadata,
            "page_content": doc.page_content
        })

        rows.append({
            "query": query,
            "stage": "bm25_compound",
            "rank": i,
            "rrf_score": None,
            "rerank_score": None,
            "dense_rank": None,
            "bm25_rank": i,
            "rrf_rank": None,
            "query_tokens": ", ".join(custom_preprocess(query)),
            "metadata": str(doc.metadata),
            "page_content": doc.page_content
        })

    # -------------------------
    # RRF 저장
    # -------------------------
    for item in rrf_docs:
        doc = item["doc"]

        comparison_results[query]["hybrid_rrf"].append({
            "rank": item["rrf_rank"],
            "rrf_score": item["rrf_score"],
            "dense_rank": item["dense_rank"],
            "bm25_rank": item["bm25_rank"],
            "metadata": doc.metadata,
            "page_content": doc.page_content
        })

        rows.append({
            "query": query,
            "stage": "hybrid_rrf",
            "rank": item["rrf_rank"],
            "rrf_score": item["rrf_score"],
            "rerank_score": None,
            "dense_rank": item["dense_rank"],
            "bm25_rank": item["bm25_rank"],
            "rrf_rank": item["rrf_rank"],
            "query_tokens": ", ".join(custom_preprocess(query)),
            "metadata": str(doc.metadata),
            "page_content": doc.page_content
        })

    # -------------------------
    # reranker 최종 저장
    # -------------------------
    for final_rank, item in enumerate(final_results, start=1):
        doc = item["doc"]

        comparison_results[query]["reranked"].append({
            "rank": final_rank,
            "rerank_score": item["rerank_score"],
            "rrf_score": item["rrf_score"],
            "rrf_rank": item["rrf_rank"],
            "dense_rank": item["dense_rank"],
            "bm25_rank": item["bm25_rank"],
            "metadata": doc.metadata,
            "page_content": doc.page_content
        })

        rows.append({
            "query": query,
            "stage": "reranked",
            "rank": final_rank,
            "rrf_score": item["rrf_score"],
            "rerank_score": item["rerank_score"],
            "dense_rank": item["dense_rank"],
            "bm25_rank": item["bm25_rank"],
            "rrf_rank": item["rrf_rank"],
            "query_tokens": ", ".join(custom_preprocess(query)),
            "metadata": str(doc.metadata),
            "page_content": doc.page_content
        })

# =========================
# DataFrame 및 CSV 저장
# =========================
df_results = pd.DataFrame(rows)

df_results.to_csv("file/csv/dense_bm25_rrf_gte_reranker_compare.csv", index=False, encoding="utf-8-sig")